## Required libraries installation

In [ ]:
%pip install accelerate scipy librosa -q
%pip install transformers torchaudio pandas datasets -q
%pip install soundfile -q
%pip install transformers==5.18.0 -q
%pip install tqdm -q

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


## All imports and custom built modules and functions

In [1]:
# Normal imports
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import gc

# Torch imports
import torch
import torchaudio
import torchaudio.transforms as T
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from transformers import ASTFeatureExtractor


/home/warden/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Device agnostic code

In [1]:
! nvidia-smi

Wed Oct 21 14:11:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 610.57.04              KMD Version: 610.57.04     CUDA UMD Version: 13.3     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 4060 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P8              2W /  115W |       4MiB /   8188MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
device = "cuda" if torch.cuda.is_available else "cpu"
device
# if it shows cpu download cuda torch audio

'cuda'

In [3]:
! python preprocessing_pipeline.py

In [5]:
! python share_split.py

Building dataset splits using pre_processing_pipeline.py...
Copying 3996 audio files to shared_forest_dataset/audio...
Done! You can now zip the 'shared_forest_dataset' folder and share it.


In [5]:
! python training.py

[INFO] Using device: cuda
[INFO] Parsing FSD50K dataset...
[INFO] Data splits created -> Train: 2796 | Val: 600 | Test: 600
[INFO] Initializing DataLoaders for 16x16 model...
[INFO] Loading AST-FastStride-16x16 architecture...
Loading weights: 100%|██████████████████████| 203/203 [00:00<00:00, 491.77it/s]
[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `527`.
[INFO] Starting training...
Epoch 1/10 [Train]:   3%|▎        | 5/174 [00:05<03:07,  1.11s/it, loss=2.1289]
Traceback (most recent call last):
  File "/home/warden/dl_project/training.py", line 128, in <module>
    train_model()
  File "/home/warden/dl_project/training.py", line 78, in train_model
    train_loss += loss.item() * inputs.size(0)
KeyboardInterrupt


# Testing on a custom .wav file

In [2]:
import torch
import torchaudio
import torchaudio.transforms as T
import torch.nn.functional as F
from transformers import ASTFeatureExtractor

# Import your model builder and configuration from the pipeline
from preprocessing_pipeline import (
    build_ast_faststride_16x16_model,
    IDX_TO_CLASS,
    SAMPLE_RATE,
    TARGET_DURATION,
)


def process_single_audio(filepath: str) -> torch.Tensor:
    """
    Replicates the exact preprocessing steps from FastStrideASTDataset.
    """
    target_samples = int(TARGET_DURATION * SAMPLE_RATE)

    # Load audio
    waveform, sr = torchaudio.load(filepath)

    # Convert to mono if necessary
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    # Resample if necessary
    if sr != SAMPLE_RATE:
        resampler = T.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
        waveform = resampler(waveform)

    waveform = waveform.squeeze(0)
    num_samples = waveform.shape[0]

    # For inference, take the center crop if it's too long
    if num_samples > target_samples:
        start = (num_samples - target_samples) // 2
        waveform = waveform[start : start + target_samples]
    elif num_samples < target_samples:
        pad = target_samples - num_samples
        waveform = F.pad(waveform, (0, pad), mode="constant", value=0.0)

    # RMS Norm + Pre-emphasis (identical to training)
    waveform = waveform - waveform.mean()
    rms = torch.sqrt(torch.mean(waveform.pow(2)) + 1e-8)
    waveform = torch.tanh(waveform * (0.1 / rms))
    waveform = torch.cat([waveform[:1], waveform[1:] - 0.97 * waveform[:-1]], dim=0)

    return waveform


def predict_wav(wav_path: str, model_weights_path: str):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print("Initializing base 16x16 AST model architecture...")
    model = build_ast_faststride_16x16_model()

    print(f"Loading trained weights from: {model_weights_path}")
    # Load the trained weights onto the appropriate device
    state_dict = torch.load(model_weights_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)

    # Set to evaluation mode and move to device
    model.eval()
    model.to(device)

    extractor = ASTFeatureExtractor.from_pretrained(
        "MIT/ast-finetuned-audioset-10-10-0.4593"
    )

    print(f"Processing audio file: {wav_path}")
    waveform = process_single_audio(wav_path)

    # Extract features using HuggingFace extractor
    inputs = extractor(waveform.numpy(), sampling_rate=SAMPLE_RATE, return_tensors="pt")
    input_values = inputs["input_values"].to(device)

    # Run Inference
    with torch.no_grad():
        logits = model(input_values).logits
        probs = torch.nn.functional.softmax(logits, dim=-1)

        top_prob, top_idx = torch.max(probs, dim=1)
        pred_class = IDX_TO_CLASS[top_idx.item()]
        confidence = top_prob.item() * 100

    print("\n--- Prediction Results ---")
    print(f"Predicted Class : {pred_class}")
    print(f"Confidence      : {confidence:.2f}%")

    print("\nFull Class Probabilities:")
    for i, prob in enumerate(probs[0]):
        print(f"{IDX_TO_CLASS[i]:>25} : {prob.item() * 100:.2f}%")


if __name__ == "__main__":
    # Point these to your specific files
    TEST_AUDIO_FILE = "chainsaw.wav"
    TRAINED_MODEL_PATH = "trained_models/best_ast_faststride_model.pth"

    predict_wav(TEST_AUDIO_FILE, TRAINED_MODEL_PATH)

Initializing base 16x16 AST model architecture...


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 15166.98it/s]
[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `527`.


Loading trained weights from: trained_models/best_ast_faststride_model.pth
Processing audio file: chainsaw.wav

--- Prediction Results ---
Predicted Class : Sawing_and_Chainsaw
Confidence      : 57.55%

Full Class Probabilities:
        Background_Nature : 6.71%
      Sawing_and_Chainsaw : 57.55%
    Gunshot_and_Explosion : 4.31%
       Engine_and_Vehicle : 21.87%
      Wood_Chop_and_Break : 3.30%
       Human_Vocalization : 3.37%
                 Dog_Bark : 2.89%


# exploring the number of parameters

In [4]:
import torch

# Import your model builder
from preprocessing_pipeline import build_ast_faststride_16x16_model


def print_model_parameters(weights_path: str):
    print("Initializing the base 16x16 AST model...")
    model = build_ast_faststride_16x16_model()

    print(f"Loading trained weights from {weights_path}...")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    state_dict = torch.load(weights_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()

    print("\n" + "=" * 80)
    print(f"{'Layer Name':<65} | {'Shape':<20} | {'Param Count':<10}")
    print("=" * 80)

    total_params = 0
    trainable_params = 0

    for name, param in model.named_parameters():
        shape_str = str(list(param.shape))
        num_params = param.numel()
        total_params += num_params
        if param.requires_grad:
            trainable_params += num_params

        print(f"{name:<65} | {shape_str:<20} | {num_params:,}")

    print("=" * 80)
    print(f"Total Parameters     : {total_params:,}")
    print(f"Trainable Parameters : {trainable_params:,}")
    print("=" * 80)


if __name__ == "__main__":
    TRAINED_MODEL_PATH = "trained_models/best_ast_faststride_model.pth"
    print_model_parameters(TRAINED_MODEL_PATH)

Initializing the base 16x16 AST model...


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 14033.09it/s]
[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `527`.


Loading trained weights from trained_models/best_ast_faststride_model.pth...

Layer Name                                                        | Shape                | Param Count
audio_spectrogram_transformer.embeddings.cls_token                | [1, 1, 768]          | 768
audio_spectrogram_transformer.embeddings.distillation_token       | [1, 1, 768]          | 768
audio_spectrogram_transformer.embeddings.position_embeddings      | [1, 514, 768]        | 394,752
audio_spectrogram_transformer.embeddings.patch_embeddings.projection.weight | [768, 1, 16, 16]     | 196,608
audio_spectrogram_transformer.embeddings.patch_embeddings.projection.bias | [768]                | 768
audio_spectrogram_transformer.layers.0.attention.q_proj.weight    | [768, 768]           | 589,824
audio_spectrogram_transformer.layers.0.attention.q_proj.bias      | [768]                | 768
audio_spectrogram_transformer.layers.0.attention.k_proj.weight    | [768, 768]           | 589,824
audio_spectrogram_transfo

# Evalvuating the model

In [1]:
import torch
import json
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Verify paths
DATASET_DIR = "shared_forest_dataset"
WEIGHTS_DIR = "trained_models"  # Update if you renamed it to 'trained_model'

print(f"[INFO] Using device: {'cuda' if torch.cuda.is_available() else 'cpu'}")
print(f"[INFO] Dataset exists: {Path(DATASET_DIR).exists()}")
print(f"[INFO] Weights directory exists: {Path(WEIGHTS_DIR).exists()}")

[INFO] Using device: cuda
[INFO] Dataset exists: True
[INFO] Weights directory exists: True


In [1]:
import os
from pathlib import Path
from evaluate_models import evaluate_single_model, RESULTS_DIR, plot_comparison_summary

# Create a symlink from 'trained_model' to 'trained_models' so evaluate_models.py finds it naturally
if not Path("trained_model").exists() and Path("trained_models").exists():
    os.symlink("trained_models", "trained_model")
    print("[INFO] Created temporary symlink: trained_model -> trained_models")

DATASET_DIR = "shared_forest_dataset"

print("\nEvaluating AST Model...")
ast_metrics, ast_targets, ast_probs = evaluate_single_model(
    model_name="ast",
    dataset_dir=DATASET_DIR,
    use_shared=True,
    batch_size=8,
    num_workers=2,
)

print("\nEvaluating MobileNetV3 Model...")
mb_metrics, mb_targets, mb_probs = evaluate_single_model(
    model_name="mobilenet",
    dataset_dir=DATASET_DIR,
    use_shared=True,
    batch_size=8,
    num_workers=2,
)

# Generate and display comparison dashboard
all_metrics = [ast_metrics, mb_metrics]
plot_comparison_summary(all_metrics, RESULTS_DIR / "model_comparison.png")

from IPython.display import Image, display

display(Image(filename=str(RESULTS_DIR / "model_comparison.png")))

/home/warden/anaconda3/envs/pytorch/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Evaluating AST Model...

██████████████████████████████████████████████████████████████████████
  EVALUATING: AST
  Device: cuda | Batch size: 8
██████████████████████████████████████████████████████████████████████

[1/4] Loading dataset...
  ├─ Validation set: 642 samples
  └─ Test set: 600 samples

[2/4] Creating dataloaders...

[3/4] Loading model...
  ├─ Building AST-FastStride-16x16 architecture...


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 16911.51it/s]


  ├─ Loading weights from: trained_model/best_ast_faststride_model.pth
  └─ Model loaded: 85,656,583 parameters
  Computing validation loss on 642 samples...

────────────────────────────────────────────────────────────
  Running inference for: AST (VALIDATION)
────────────────────────────────────────────────────────────


  Evaluating ast (validation):   0%|                              | 0/81 [00:00<?, ?batch/s]


TypeError: cross_entropy_loss(): argument 'input' (position 1) must be Tensor, not SequenceClassifierOutput

In [7]:
%run evaluate_models.py --model all --dataset-dir ./data


══════════════════════════════════════════════════════════════════════
  🌿 EcoAST — Model Evaluation Pipeline
  2026-10-27 12:53:08
══════════════════════════════════════════════════════════════════════

██████████████████████████████████████████████████████████████████████
  EVALUATING: AST
  Device: cuda | Batch size: 8
██████████████████████████████████████████████████████████████████████

[1/4] Loading dataset...
  ├─ Validation set: 600 samples
  └─ Test set: 600 samples

[2/4] Creating dataloaders...

[3/4] Loading model...
  ├─ Building AST-FastStride-16x16 architecture...


Loading weights: 100%|██████████| 203/203 [00:00<00:00, 718.10it/s]


  ├─ Loading weights from: trained_model/best_ast_faststride_model.pth
  └─ Model loaded: 85,656,583 parameters
  Computing validation loss on 600 samples...

────────────────────────────────────────────────────────────
  Running inference for: AST (VALIDATION)
────────────────────────────────────────────────────────────


  Evaluating ast (validation): 100%|██████████████████████████████| 75/75 [00:14<00:00,  5.11batch/s]



────────────────────────────────────────────────────────────
  Running inference for: AST
────────────────────────────────────────────────────────────


...


  ⏱  Inference completed in 5.0s (119.6 samples/sec)

[4/4] Computing metrics...

══════════════════════════════════════════════════════════════════════
  EVALUATION RESULTS: MOBILENET
══════════════════════════════════════════════════════════════════════
  Validation Loss    : 2.2416
  Test Loss          : 2.0968
  Accuracy           : 39.00%
  mAP (macro)        : 0.4500
  ROC-AUC (macro)    : 0.7408
  Macro F1           : 0.3515
  Weighted F1        : 0.3827
  Macro F2 (threats) : 0.3278
──────────────────────────────────────────────────────────────────────

  Per-Class Average Precision (AP):
    Background_Nature         : 0.6646
    Sawing_and_Chainsaw       : 0.2511 ⚠ THREAT
    Gunshot_and_Explosion     : 0.6944 ⚠ THREAT
    Engine_and_Vehicle        : 0.4286 ⚠ THREAT
    Wood_Chop_and_Break       : 0.2448
    Human_Vocalization        : 0.3616
    Dog_Bark                  : 0.5051
    ────────────────────────────────────────
    mAP                       : 0.4500

  Per-Clas

  Evaluating effnet (validation): 100%|██████████████████████████████| 75/75 [00:05<00:00, 12.62batch/s]



────────────────────────────────────────────────────────────
  Running inference for: EFFNET
────────────────────────────────────────────────────────────


  Evaluating effnet: 100%|██████████████████████████████| 75/75 [00:06<00:00, 12.48batch/s]



  ⏱  Inference completed in 6.0s (99.8 samples/sec)

[4/4] Computing metrics...

══════════════════════════════════════════════════════════════════════
  EVALUATION RESULTS: EFFNET
══════════════════════════════════════════════════════════════════════
  Validation Loss    : 4.4307
  Test Loss          : 4.3381
  Accuracy           : 24.00%
  mAP (macro)        : 0.3123
  ROC-AUC (macro)    : 0.6707
  Macro F1           : 0.2044
  Weighted F1        : 0.1776
  Macro F2 (threats) : 0.2316
──────────────────────────────────────────────────────────────────────

  Per-Class Average Precision (AP):
    Background_Nature         : 0.7496
    Sawing_and_Chainsaw       : 0.5353 ⚠ THREAT
    Gunshot_and_Explosion     : 0.1773 ⚠ THREAT
    Engine_and_Vehicle        : 0.1526 ⚠ THREAT
    Wood_Chop_and_Break       : 0.1164
    Human_Vocalization        : 0.1964
    Dog_Bark                  : 0.2586
    ────────────────────────────────────────
    mAP                       : 0.3123

  Per-Class F2